# Exact match or string matching measure (using fuzzy logic)

In [ ]:
from google.colab import drive;
drive.mount('/content/drive');
!pip install pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 30.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.2
    Uninstalling scikit-learn-1.3.2:
      Successfully uninstalled scikit-learn-1.3.2


In [ ]:
!pip install fuzzywuzzy pandas openpyxl

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/Colab Notebooks/IST/Experiment 1/GPT_4_Turbo/Result/R1.xlsx'
df = pd.read_excel(file_path)
print(df.head())

     Run                                            ACAS XU  \
0  Run 1  G1: Ensure the ACAS Xu system is secure agains...   
1  Run 2  Goal1: Ensure ACAS Xu system is reliable and s...   
2  Run 3  G1: Ensure ACAS Xu operates securely to preven...   
3  Run 4  Goal 1: Ensure overall security of ACAS Xu\nGo...   
4  Run 5  G1: Ensure that ACAS Xu system is secure again...   

                                            BlueROV2  \
0  Goal1: To demonstrate the operational reliabil...   
1  Goal: Ensure the overall safety and effectiven...   
2  G1: Ensure BlueROV2 operates safely and reliab...   
3  G1: Ensure that the BlueROV2 system is safe fo...   
4  G1: Ensure the Safety of the BlueROV2 System\n...   

                                                GPCA  \
0  G1: Ensure GPCA System is Safe for Patient Use...   
1  G1: Ensure that the GPCA system is safe for pa...   
2  G1: Ensure that the GPCA system is safe for pa...   
3  G1: Ensure the GPCA system safely administers ...   
4  G

In [ ]:
file_path2 = '/content/drive/MyDrive/Colab Notebooks/IST/Experiment 1/Ground_Truth_AC.xlsx'
ground_truth_df = pd.read_excel(file_path2)
print(ground_truth_df.head())

          Case                                                 AC
0      ACAS XU  G0: ACAS Xu satisfies security requirements\nG...
1     BlueROV2  G1: Hazards for BlueROV2 addressed in accordan...
2         GPCA  G1: Operational safety is verified in "GPCA sy...
3  IM_Software  G1: IM server software is acceptable secure in...
4     DeepMind  G1: Interpretability Claim - The ML system is ...


In [ ]:
from fuzzywuzzy import fuzz


for index, experiment_row in df.iterrows():
    for case in ['ACAS XU', 'BlueROV2', 'GPCA', 'IM_Software', 'DeepMind']:
        experiment_text = experiment_row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]
        similarity_score = fuzz.ratio(experiment_text, ground_truth_text)/ 100.0

        print(f"Similarity score for {case} in {experiment_row['Run']}: {similarity_score}")
    print("-------")

Similarity score for ACAS XU in Run 1: 0.04
Similarity score for BlueROV2 in Run 1: 0.02
Similarity score for GPCA in Run 1: 0.04
Similarity score for IM_Software in Run 1: 0.04
Similarity score for DeepMind in Run 1: 0.05
-------
Similarity score for ACAS XU in Run 2: 0.03
Similarity score for BlueROV2 in Run 2: 0.02
Similarity score for GPCA in Run 2: 0.04
Similarity score for IM_Software in Run 2: 0.05


/usr/local/lib/python3.10/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Similarity score for DeepMind in Run 2: 0.05
-------
Similarity score for ACAS XU in Run 3: 0.02
Similarity score for BlueROV2 in Run 3: 0.02
Similarity score for GPCA in Run 3: 0.04
Similarity score for IM_Software in Run 3: 0.03
Similarity score for DeepMind in Run 3: 0.05
-------
Similarity score for ACAS XU in Run 4: 0.03
Similarity score for BlueROV2 in Run 4: 0.04
Similarity score for GPCA in Run 4: 0.04
Similarity score for IM_Software in Run 4: 0.05
Similarity score for DeepMind in Run 4: 0.07
-------
Similarity score for ACAS XU in Run 5: 0.03
Similarity score for BlueROV2 in Run 5: 0.06
Similarity score for GPCA in Run 5: 0.05
Similarity score for IM_Software in Run 5: 0.05
Similarity score for DeepMind in Run 5: 0.01
-------


# Bleu score

In [ ]:
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 957.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 2.1 MB/s eta 0:00:00


In [ ]:
import sacrebleu

def calculate_bleu_score(experiment_text, ground_truth_text):
    references = [[ground_truth_text]]
    system_output = [experiment_text]
    bleu = sacrebleu.corpus_bleu(system_output, references)
    # Scale the score to a range between 0 and 1 and round to two decimal points
    return round(bleu.score / 100, 2)

for case in ['ACAS XU', 'BlueROV2', 'GPCA', 'IM_Software', 'DeepMind']:
    print(f"Calculating BLEU scores for {case}:")
    for index, row in df.iterrows():
        experiment_text = row[case]
        ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

        if pd.notnull(experiment_text) and pd.notnull(ground_truth_text):
            bleu_score = calculate_bleu_score(experiment_text, ground_truth_text)
            print(f"Run {index + 1} BLEU score: {bleu_score}")
        else:
            print(f"Run {index + 1} BLEU score: Data Missing")
    print("-------")


Calculating BLEU scores for ACAS XU:
Run 1 BLEU score: 0.03
Run 2 BLEU score: 0.03
Run 3 BLEU score: 0.02
Run 4 BLEU score: 0.01
Run 5 BLEU score: 0.02
-------
Calculating BLEU scores for BlueROV2:
Run 1 BLEU score: 0.0
Run 2 BLEU score: 0.0
Run 3 BLEU score: 0.01
Run 4 BLEU score: 0.01
Run 5 BLEU score: 0.01
-------
Calculating BLEU scores for GPCA:
Run 1 BLEU score: 0.01
Run 2 BLEU score: 0.01
Run 3 BLEU score: 0.01
Run 4 BLEU score: 0.01
Run 5 BLEU score: 0.01
-------
Calculating BLEU scores for IM_Software:
Run 1 BLEU score: 0.01
Run 2 BLEU score: 0.01
Run 3 BLEU score: 0.01
Run 4 BLEU score: 0.0
Run 5 BLEU score: 0.01
-------
Calculating BLEU scores for DeepMind:
Run 1 BLEU score: 0.02
Run 2 BLEU score: 0.01
Run 3 BLEU score: 0.01
Run 4 BLEU score: 0.01
Run 5 BLEU score: 0.0
-------


# Semantic Similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

for case in ['ACAS XU', 'BlueROV2', 'GPCA', 'IM_Software', 'DeepMind']:
    experiment_texts = df[case].tolist()
    ground_truth_text = ground_truth_df.loc[ground_truth_df['Case'] == case, 'AC'].iloc[0]

    combined_texts = experiment_texts + [ground_truth_text]

    tfidf_matrix = vectorizer.fit_transform(combined_texts)

    experiment_vectors = tfidf_matrix[:-1]
    ground_truth_vector = tfidf_matrix[-1:]

    cosine_similarities = cosine_similarity(experiment_vectors, ground_truth_vector)

    for i, similarity_score in enumerate(cosine_similarities.flatten(), 1):
        # Round the score to two decimal places without multiplying by 100
        rounded_score = round(similarity_score, 2)
        print(f"Cosine similarity for {case}, Run {i}: {rounded_score}")
    print("-------")


Cosine similarity for ACAS XU, Run 1: 0.33
Cosine similarity for ACAS XU, Run 2: 0.22
Cosine similarity for ACAS XU, Run 3: 0.19
Cosine similarity for ACAS XU, Run 4: 0.14
Cosine similarity for ACAS XU, Run 5: 0.29
-------
Cosine similarity for BlueROV2, Run 1: 0.05
Cosine similarity for BlueROV2, Run 2: 0.07
Cosine similarity for BlueROV2, Run 3: 0.07
Cosine similarity for BlueROV2, Run 4: 0.15
Cosine similarity for BlueROV2, Run 5: 0.09
-------
Cosine similarity for GPCA, Run 1: 0.06
Cosine similarity for GPCA, Run 2: 0.14
Cosine similarity for GPCA, Run 3: 0.11
Cosine similarity for GPCA, Run 4: 0.07
Cosine similarity for GPCA, Run 5: 0.16
-------
Cosine similarity for IM_Software, Run 1: 0.18
Cosine similarity for IM_Software, Run 2: 0.15
Cosine similarity for IM_Software, Run 3: 0.1
Cosine similarity for IM_Software, Run 4: 0.09
Cosine similarity for IM_Software, Run 5: 0.13
-------
Cosine similarity for DeepMind, Run 1: 0.21
Cosine similarity for DeepMind, Run 2: 0.22
Cosine simi